## Evaluate Presidio Analyzer with GLiNER

This notebook shows how to use `nlp_gliner.py` style wiring in the Presidio evaluator flow.

Notes:
- This notebook uses the custom `GLiNERNlpEngine` and `GLiNERRecognizer` from `gliner_engine.py`.
- The local model path defaults to `/home/capcom/models/gliner-multi-edu`.
- GLiNER on CPU is much slower than the HanLP example, so evaluation uses a small subset by default.


In [50]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [51]:
from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json

from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.experiment_tracking import get_experiment_tracker
from presidio_evaluator.models import PresidioAnalyzerWrapper

from presidio_analyzer import AnalyzerEngine, RecognizerRegistry

from gliner_engine import GLiNERNlpEngine, GLiNERRecognizer, GLINER_LABEL_MAPPING

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

%reload_ext autoreload
%autoreload 2

## 1. Load dataset

In [52]:
dataset_name = "data_person_1000_zh_fixed_age.json"
dataset = InputSample.read_dataset_json(
    Path(Path.cwd(), "data", dataset_name),
    token_model_version="zh_core_web_sm",
)
print(len(dataset))
dataset[0]

tokenizing input: 100%|██████████| 975/975 [00:15<00:00, 62.49it/s]

975


Full text: 白雅宁是一位43岁的女性口腔卫生师，现居住于黑龙江省哈尔滨市南岗区中山路123号，可通过邮箱baiyaning@163.com或手机13945671234联系。她的身份证号码为230103198008273629。近期她出现不明肿块、持续疲劳和体重下降等症状，经诊断为癌症。目前正在韩雪梅医生的指导下使用青霉素进行治疗。白雅宁的信用评分为850分，年收入为56万元人民币。最近的交易记录包括一笔央行内部资金划转。
Spans: [Span(type: PERSON, value: 白雅宁, char_span: [0: 3]), Span(type: AGE, value: 43岁, char_span: [6: 9]), Span(type: STREET_ADDRESS, value: 黑龙江省哈尔滨市南岗区中山路123号, char_span: [22: 40]), Span(type: EMAIL_ADDRESS, value: baiyaning@163.com, char_span: [46: 63]), Span(type: PHONE_NUMBER, value: 13945671234, char_span: [66: 77]), Span(type: PERSON, value: 韩雪梅, char_span: [141: 144]), Span(type: PERSON, value: 白雅宁, char_span: [160: 163])]

In [53]:
def get_entity_counts(dataset: List[InputSample]) -> Dict:
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter

entity_counts = get_entity_counts(dataset)
pprint(entity_counts.most_common(), compact=True)

[('O', 79696), ('STREET_ADDRESS', 5497), ('PERSON', 3394),
 ('EMAIL_ADDRESS', 3385), ('AGE', 1925), ('PHONE_NUMBER', 974),
 ('CREDIT_CARD', 53)]


## 2. Create GLiNER-based AnalyzerEngine

In [54]:
MODEL_NAME = "Ihor/gliner-multi-edu"
THRESHOLD = 0.5
LABELS = [
    "person",
    "location",
    "organization",
    "age",
    "email address",
    "phone number",
    "personal identification number",
]

nlp_engine = GLiNERNlpEngine(
    model_name=MODEL_NAME,
    labels=LABELS,
    label_mapping=GLINER_LABEL_MAPPING,
    threshold=THRESHOLD,
)
recognizer = GLiNERRecognizer(nlp_engine=nlp_engine, score_threshold=THRESHOLD)

registry = RecognizerRegistry(supported_languages=["zh"])
registry.add_recognizer(recognizer)

analyzer_engine = AnalyzerEngine(
    nlp_engine=nlp_engine,
    registry=registry,
    supported_languages=["zh"],
)

pprint(analyzer_engine.get_supported_entities("zh"), compact=True)
pprint([rec.name for rec in analyzer_engine.registry.get_recognizers("zh", all_fields=True)], compact=True)

Loading GLiNER model from Ihor/gliner-multi-edu on CPU...


/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
2026-03-19 20:29:27 ERROR: Cannot load model from /Users/jyu36/Library/Caches/stanza/1.11.0/resources/en/tokenize/combined.pt


GLiNER model loaded in 105.8s
['INCOME', 'LOCATION', 'MEDICATION', 'TRANSACTION', 'PHONE_NUMBER',
 'ORGANIZATION', 'CREDIT_SCORE', 'DIAGNOSIS', 'PERSON', 'EMAIL_ADDRESS', 'AGE',
 'PROFESSION', 'GENDER', 'ID']
['GLiNERRecognizer']


## 3. Quick single-text test

In [55]:
text = (
    "白雅宁是一位43岁的女性口腔卫生师，现居住于黑龙江省哈尔滨市南岗区中山路123号，"
    "可通过邮箱baiyaning@163.com或手机13945671234联系。她的身份证号码为230103198008273629。"
    "近期她出现不明肿块、持续疲劳和体重下降等症状，经诊断为癌症。"
)

res = analyzer_engine.analyze(text=text, language="zh")
for result in res:
    print(result.entity_type, result.start, result.end, text[result.start:result.end], result.score)

Running GLiNER inference...
GLiNER inference finished in 0.4s
PHONE_NUMBER 66 77 13945671234 0.962668240070343
PERSON 0 3 白雅宁 0.9374294281005859
AGE 6 8 43 0.6840072274208069


## 4. Align entity names

Presidio evaluator expects the gold labels and predicted labels to live in the same entity space.


In [ ]:
entities_mapping = PresidioAnalyzerWrapper.presidio_entities_map
pprint(entities_mapping, compact=True)

aligned_dataset = SpanEvaluator.align_entity_types(
    dataset,
    entities_mapping=entities_mapping,
    allow_missing_mappings=True,
)
new_entity_counts = get_entity_counts(aligned_dataset)
pprint(new_entity_counts.most_common(), compact=True)

{'ADDRESS': 'LOCATION',
 'AGE': 'AGE',
 'BIRTHDAY': 'DATE_TIME',
 'CITY': 'LOCATION',
 'CREDIT_CARD': 'CREDIT_CARD',
 'CREDIT_CARD_NUMBER': 'CREDIT_CARD',
 'DATE': 'DATE_TIME',
 'DATE_OF_BIRTH': 'DATE_TIME',
 'DATE_TIME': 'DATE_TIME',
 'DOB': 'DATE_TIME',
 'DOMAIN': 'URL',
 'DOMAIN_NAME': 'URL',
 'EMAIL': 'EMAIL_ADDRESS',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'FACILITY': 'LOCATION',
 'FIRST_NAME': 'PERSON',
 'GPE': 'LOCATION',
 'HCW': 'PERSON',
 'HOSP': 'ORGANIZATION',
 'HOSPITAL': 'ORGANIZATION',
 'IBAN': 'IBAN_CODE',
 'IBAN_CODE': 'IBAN_CODE',
 'ID': 'ID',
 'IP_ADDRESS': 'IP_ADDRESS',
 'LAST_NAME': 'PERSON',
 'LOC': 'LOCATION',
 'LOCATION': 'LOCATION',
 'NAME': 'PERSON',
 'NATIONALITY': 'NRP',
 'NORP': 'NRP',
 'NRP': 'NRP',
 'O': 'O',
 'ORG': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PATIENT': 'PERSON',
 'PATORG': 'ORGANIZATION',
 'PER': 'PERSON',
 'PERSON': 'PERSON',
 'PHONE': 'PHONE_NUMBER',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'PREFIX': 'TITLE',
 'SSN': 'US_SSN',
 'STAFF': 'PE

## 5. Evaluate on a small subset

GLiNER is slow on CPU. Start with a small subset and scale only if results look reasonable.


In [57]:
subset_size = 50
filtered_dataset = aligned_dataset[:subset_size]
print(f"Evaluating {len(filtered_dataset)} samples")

experiment = get_experiment_tracker()
evaluator = SpanEvaluator(model=analyzer_engine)

params = {
    "dataset_name": dataset_name,
    "subset_size": subset_size,
    "model_name": evaluator.model.name,
    "gliner_model_path": MODEL_NAME,
    "threshold": THRESHOLD,
    "labels": LABELS,
}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(filtered_dataset)
experiment.log_parameter("entity_mappings", json.dumps(entities_mapping))

Evaluating 50 samples
--------
Entities supported by this Presidio Analyzer instance:
INCOME, LOCATION, MEDICATION, TRANSACTION, PHONE_NUMBER, ORGANIZATION, CREDIT_SCORE, DIAGNOSIS, PERSON, EMAIL_ADDRESS, AGE, PROFESSION, GENDER, ID


/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/presidio_evaluator/evaluation/base_evaluator.py:83: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn("skip words not provided, using default skip words. "


In [58]:
%%time
evaluation_results = evaluator.evaluate_all(filtered_dataset)
results = evaluator.calculate_score(evaluation_results)

experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
experiment.end()

pprint({
    "PII F": results.pii_f,
    "PII recall": results.pii_recall,
    "PII precision": results.pii_precision,
})

Running model PresidioAnalyzerWrapper on dataset...
Running GLiNER inference...
GLiNER inference finished in 0.5s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.5s
Running GLiNER inference...
GLiNER inference finished in 0.5s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.5s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inference...
GLiNER inference finished in 0.4s
Running GLiNER inf

## 6. Inspect examples and errors

In [59]:
sample = filtered_dataset[0]
print(sample.full_text)
print("\nGOLD SPANS:")
for s in sample.spans:
    print(s.entity_type, s.start_position, s.end_position, sample.full_text[s.start_position:s.end_position])

print("\nPRED SPANS:")
preds = analyzer_engine.analyze(text=sample.full_text, language="zh")
for p in preds:
    print(p.entity_type, p.start, p.end, sample.full_text[p.start:p.end], p.score)

白雅宁是一位43岁的女性口腔卫生师，现居住于黑龙江省哈尔滨市南岗区中山路123号，可通过邮箱baiyaning@163.com或手机13945671234联系。她的身份证号码为230103198008273629。近期她出现不明肿块、持续疲劳和体重下降等症状，经诊断为癌症。目前正在韩雪梅医生的指导下使用青霉素进行治疗。白雅宁的信用评分为850分，年收入为56万元人民币。最近的交易记录包括一笔央行内部资金划转。

GOLD SPANS:
PERSON 0 3 白雅宁
AGE 6 9 43岁
LOCATION 22 40 黑龙江省哈尔滨市南岗区中山路123号
EMAIL_ADDRESS 46 63 baiyaning@163.com
PHONE_NUMBER 66 77 13945671234
PERSON 141 144 韩雪梅
PERSON 160 163 白雅宁

PRED SPANS:
Running GLiNER inference...
GLiNER inference finished in 0.4s
PHONE_NUMBER 66 77 13945671234 0.9552612900733948
PERSON 0 3 白雅宁 0.9106149077415466
PERSON 141 144 韩雪梅 0.7266722321510315
AGE 6 9 43岁 0.6706998348236084
PERSON 160 162 白雅 0.5583215355873108


In [60]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('中国 工商 银行', 11),
 ('岁', 3),
 ('印度 法证 服务', 2),
 ('林雨 晴 张雪梅', 1),
 ('戴雪梅 柏晓', 1),
 ('张建国 贾思', 1),
 ('印度 法证 服务 公司', 1),
 ('卢柯 ，', 1),
 ('阿司匹林', 1),
 ('高丽华 医生', 1)]
---------------
Example sentence with each FP token:
	- 中国 工商 银行 (`中国 工商 银行` pred as ORGANIZATION)
	- 岁 (`岁` pred as O)
	- 印度 法证 服务 (`印度 法证 服务` pred as ORGANIZATION)
	- 林雨 晴 张雪梅 (`林雨 晴 张雪梅` pred as O)
	- 戴雪梅 柏晓 (`戴雪梅 柏晓` pred as O)
	- 张建国 贾思 (`张建国 贾思` pred as O)
	- 印度 法证 服务 公司 (`印度 法证 服务 公司` pred as ORGANIZATION)
	- 卢柯 ， (`卢柯 ，` pred as O)
	- 阿司匹林 (`阿司匹林` pred as PERSON)
	- 高丽华 医生 (`高丽华 医生` pred as O)


[('中国 工商 银行', 11),
 ('岁', 3),
 ('印度 法证 服务', 2),
 ('林雨 晴 张雪梅', 1),
 ('戴雪梅 柏晓', 1),
 ('张建国 贾思', 1),
 ('印度 法证 服务 公司', 1),
 ('卢柯 ，', 1),
 ('阿司匹林', 1),
 ('高丽华 医生', 1)]

## 7. Plot results

In [61]:
plotter = Plotter(
    results=results,
    model_name=evaluator.model.name,
    save_as="svg",
    beta=2,
)
output_folder = Path(Path.cwd(), "plotter_output_gliner")
plotter.plot_scores(output_folder=output_folder)
output_folder

PosixPath('/Users/jyu36/code/ic-llm/presidio/plotter_output_gliner')

In [62]:
ModelError.most_common_fp_tokens(results.model_errors)

Most common false positive tokens:
[('中国 工商 银行', 11),
 ('岁', 3),
 ('印度 法证 服务', 2),
 ('林雨 晴 张雪梅', 1),
 ('戴雪梅 柏晓', 1),
 ('张建国 贾思', 1),
 ('印度 法证 服务 公司', 1),
 ('卢柯 ，', 1),
 ('阿司匹林', 1),
 ('高丽华 医生', 1)]
---------------
Example sentence with each FP token:
	- 中国 工商 银行 (`中国 工商 银行` pred as ORGANIZATION)
	- 岁 (`岁` pred as O)
	- 印度 法证 服务 (`印度 法证 服务` pred as ORGANIZATION)
	- 林雨 晴 张雪梅 (`林雨 晴 张雪梅` pred as O)
	- 戴雪梅 柏晓 (`戴雪梅 柏晓` pred as O)
	- 张建国 贾思 (`张建国 贾思` pred as O)
	- 印度 法证 服务 公司 (`印度 法证 服务 公司` pred as ORGANIZATION)
	- 卢柯 ， (`卢柯 ，` pred as O)
	- 阿司匹林 (`阿司匹林` pred as PERSON)
	- 高丽华 医生 (`高丽华 医生` pred as O)


[('中国 工商 银行', 11),
 ('岁', 3),
 ('印度 法证 服务', 2),
 ('林雨 晴 张雪梅', 1),
 ('戴雪梅 柏晓', 1),
 ('张建国 贾思', 1),
 ('印度 法证 服务 公司', 1),
 ('卢柯 ，', 1),
 ('阿司匹林', 1),
 ('高丽华 医生', 1)]

In [63]:
fps_df = ModelError.get_fps_dataframe(results.model_errors, entity=["PERSON"])
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,阿司匹林,阿司匹林,O,PERSON


In [64]:
fps_df = ModelError.get_fps_dataframe(results.model_errors)
fps_df[["full_text", "token", "annotation", "prediction"]]

,full_text,token,annotation,prediction
0,印度 法证 服务,印度 法证 服务,O,ORGANIZATION
1,林雨 晴 张雪梅,林雨 晴 张雪梅,PERSON,O
2,戴雪梅 柏晓,戴雪梅 柏晓,PERSON,O
3,张建国 贾思,张建国 贾思,PERSON,O
4,印度 法证 服务 公司,印度 法证 服务 公司,O,ORGANIZATION
5,印度 法证 服务,印度 法证 服务,O,ORGANIZATION
6,中国 工商 银行,中国 工商 银行,O,ORGANIZATION
7,卢柯 ，,卢柯 ，,PERSON,O
8,岁,岁,AGE,O
9,阿司匹林,阿司匹林,O,PERSON


In [65]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)

Most common false negative tokens:
[('shenyutong 163.com', 4),
 ('163.com', 2),
 ('34 岁', 2),
 ('浙江省 杭州市 西湖区 文三路 456号', 2),
 ('黑龙江省 哈尔滨市 南岗区 中山路 123号', 1),
 ('baiyaning 163.com', 1),
 ('浙江省 杭州市 西湖区 文三路 826号', 1),
 ('taolixuan 163.com', 1),
 ('江苏省 南京市 鼓楼区 中山 北路 1890号', 1),
 ('林雨 晴 张雪梅 林雨 晴年', 1),
 ('浙江省 杭州市 西湖区 文三路 3763号', 1),
 ('shokma 5698 163.com', 1),
 ('戴雪梅 柏晓 岚年', 1),
 ('黑龙江省 哈尔滨市 南岗区 学府路 4188号', 1),
 ('jiasiming 163.com', 1)]
---------------
Example sentence with each FN token:
	- shenyutong @ 163.com (`shenyutong 163.com` annotated as O)
	- @ 163.com (`163.com` annotated as O)
	- 34 岁 (`34 岁` annotated as O)
	- 浙江省 杭州市 西湖区 文三路 456号 (`浙江省 杭州市 西湖区 文三路 456号` annotated as O)
	- 黑龙江省 哈尔滨市 南岗区 中山路 123号 (`黑龙江省 哈尔滨市 南岗区 中山路 123号` annotated as O)
	- baiyaning @ 163.com (`baiyaning 163.com` annotated as O)
	- 浙江省 杭州市 西湖区 文三路 826号 (`浙江省 杭州市 西湖区 文三路 826号` annotated as O)
	- taolixuan @ 163.com (`taolixuan 163.com` annotated as O)
	- 江苏省 南京市 鼓楼区 中山 北路 1890号 (`江苏省 南京市 鼓楼区 中山 北路 1890号` annotat

[('shenyutong 163.com', 4),
 ('163.com', 2),
 ('34 岁', 2),
 ('浙江省 杭州市 西湖区 文三路 456号', 2),
 ('黑龙江省 哈尔滨市 南岗区 中山路 123号', 1),
 ('baiyaning 163.com', 1),
 ('浙江省 杭州市 西湖区 文三路 826号', 1),
 ('taolixuan 163.com', 1),
 ('江苏省 南京市 鼓楼区 中山 北路 1890号', 1),
 ('林雨 晴 张雪梅 林雨 晴年', 1),
 ('浙江省 杭州市 西湖区 文三路 3763号', 1),
 ('shokma 5698 163.com', 1),
 ('戴雪梅 柏晓 岚年', 1),
 ('黑龙江省 哈尔滨市 南岗区 学府路 4188号', 1),
 ('jiasiming 163.com', 1)]

In [66]:
fns_df = ModelError.get_fns_dataframe(results.model_errors)
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,黑龙江省 哈尔滨市 南岗区 中山路 123号,黑龙江省 哈尔滨市 南岗区 中山路 123号,O,LOCATION
1,baiyaning @ 163.com,baiyaning 163.com,O,EMAIL_ADDRESS
2,浙江省 杭州市 西湖区 文三路 826号,浙江省 杭州市 西湖区 文三路 826号,O,LOCATION
3,taolixuan @ 163.com,taolixuan 163.com,O,EMAIL_ADDRESS
4,江苏省 南京市 鼓楼区 中山 北路 1890号,江苏省 南京市 鼓楼区 中山 北路 1890号,O,LOCATION
5,@ 163.com,163.com,O,EMAIL_ADDRESS
6,林雨 晴 张雪梅 林雨 晴年,林雨 晴 张雪梅 林雨 晴年,O,PERSON
7,浙江省 杭州市 西湖区 文三路 3763号,浙江省 杭州市 西湖区 文三路 3763号,O,LOCATION
8,a shokma 5698 @ 163.com,shokma 5698 163.com,O,EMAIL_ADDRESS
9,戴雪梅 柏晓 岚年,戴雪梅 柏晓 岚年,O,PERSON
